# Practice 1: Full Adder in PYNQ

PYNQ lets Python on the Kria KV260’s Arm processor program the FPGA and communicate with the circuit running there. In this practice, Python drives the three inputs of your full adder through AXI GPIO and reads back the sum and carry bits.

## 1. Download your bitstream to the FPGA

Place your Vivado `.bit` file and its matching `.hwh` file in the `Overlays` folder with the same base name. Enter your bitstream’s filename below, including `.bit`, and run the cell with **Shift + Enter**. This programs the FPGA. Once the cell prints **Loaded**, continue to the test.

In [ ]:
from pathlib import Path
from full_adder import load_adder, print_test

bitstream_name = "your_design.bit"  # Replace with your filename
adder = None
adder = load_adder(Path("Overlays") / bitstream_name)
print(f"Loaded {bitstream_name}")

## 2. Test your full adder

For the first test, **1 + 0 + 1 = 2**, or **10** in binary. Predict the carry and sum bits, then run the cell and compare. Change any input to 0 or 1 and rerun just this cell for another test. Run the download cell again if you choose another design or restart the notebook kernel.

In [ ]:
a, b, cin = 1, 0, 1
passed = print_test(adder, a, b, cin)

## 3. Clean Up

Once you are finished with the Overlay, free the resources for the next part of the mini project and delete the Full Adder Class when finished.

In [ ]:
import gc

adder.free()
del adder
gc.collect()

# Practice 2: 2D Convolution on the Kria KV260
## Python → C → ARM64 Assembly

Run the same 3 × 3 convolution on your photo using Python, C, and AArch64 assembly on the KV260’s Arm processor. 
The implementations are supplied. Run each version, check its output, and record its time and one observation. 
Later you will run the supplied row-order C function on MicroBlaze V.

| Part | Level | What you do |
| --- | --- | --- |
| 1 | The idea | See a convolution on Mario, compute one pixel by hand, take your photo |
| 2 | Python → C | Measure Python and C v1, and explain the difference |
| 3 | C | **Measure C v2** (loops in another order) and compare its time with C v1 |
| 4 | C → assembly | Translate RISC-V (Ripes) to ARM64 |
| 5 | Assembly | Measure ASM v1, a direct translation of C v2 |
| 6–8 | Assembly | Compare pointers (v2), multiply-accumulate (v3), and NEON (v4) |
| 9–10 | All | Compare every version, watch the race |
| 11 | C | **Measure C v2** on the MicroBlaze CPU and compare to ARM cores |
| 12 | DPU | Classify a video with the DPU accelerator |

### How to use this notebook

- Run a cell with **Shift + Enter**. Run the cells **in order**, from top to bottom.
- The questions are the cells marked **Q#**: **Q** stands for *question*, and # is its
  number (**Q1**,**Q2**, … **Q9**). That cell only asks the question.
- **YOUR ANSWER GOES IN THE CELL IMMEDIATELY BELOW THE QUESTION**, marked
  **ANSWER Q# HERE** (for example, **ANSWER Q1 HERE** right below **Q1**). Double-click
  it, replace the line in *italics* with your answer, then press Shift + Enter. Do not
  write your answers in the question cells.
- Save often (**Ctrl + S**).

---
## Part 0. Check the setup

The cell below builds the C and assembly code with `make`,
and runs a correctness test: every version must give exactly the same result 
as Python on 13 image sizes.

**Expected:** The build, then a line ending in `all OK`. The versions marked **(yours)** are the ones
you write later: `C v2 rows (yours)` and `ASM v1 brute (yours)` say `MISMATCH` until you
write them in Parts 3 and 5. `ASM v3 MAC (yours)` already says `OK`: it starts as a working
copy of ASM v2, and in Part 7 you make it use fewer instructions.

In [ ]:
!uname -m
!make -B all
!python3 conv_bench.py --test

---
## Part 1. What is a convolution?

A **kernel** is a small grid of numbers, here 3 × 3. It slides over the image. At each
position, every pixel under the kernel is multiplied by the kernel number on top of it, and
the 9 products are added. That sum is one pixel of the output:

$$
\text{out}[y][x] = \sum_{k_y=0}^{2}\ \sum_{k_x=0}^{2}\ \text{in}[y+k_y][x+k_x] \cdot k[k_y][k_x]
$$

The kernel never hangs outside the image, so the output is 2 pixels smaller in each
direction. Image filters (blur, edge detection) and the neural networks that recognize
images are built on exactly this operation.

Run the cell below to watch it, on a pixel-art **Mario**. The convolution only sees
numbers, so each colour is one gray number (the legend under the animation says which).
The yellow 3 × 3 window slides over the image from left to right and from top to bottom;
at each position the 9 products are added and one output pixel appears. The kernel is the
**edge-detection kernel** you will use on your photo: where the colour does not change the
result is 0, and where it changes it is large, so the output draws Mario's outline. Put
the mouse on the animation to pause it. It plays once; to watch it again, run the cell
again. For another view,
watch [this animated convolution](https://www.youtube.com/watch?v=xjqCTp4xAtA).

In [ ]:
from conv_demo import notebook_animation, notebook_image, notebook_race, notebook_step, import_photo
from camonly import capture_image, cleanup_device_trees
from pynq import Overlay

OVERLAY_DIR = "/root/jupyter_notebooks/YOLO"

cleanup_device_trees()

overlay = Overlay(f"{OVERLAY_DIR}/kv260-camonly.bit",
                    dtbo=f"{OVERLAY_DIR}/kv260-camonly.dtbo", ignore_version=True)

notebook_animation()

Now look at one position in detail, with a smaller image of random numbers and another
kernel. The cell below shows how output pixel **(0, 0)** is computed: the 9 numbers of the
yellow window are multiplied by the kernel and summed.

In [ ]:
notebook_step(0, 0)

**Q1.** Compute output pixel **out[1][2] by hand** for this new image and kernel. Write the
9 products and the result. *(Rows and columns are numbered from 0. The window for
out[1][2] starts at row 1, column 2 of the input.)*

**Input image** (5 × 6)

| | col 0 | col 1 | col 2 | col 3 | col 4 | col 5 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| **row 0** | 8 | 6 | 2 | 2 | 6 | 6 |
| **row 1** | 5 | 8 | 9 | 6 | 5 | 2 |
| **row 2** | 4 | 7 | 7 | 9 | 5 | 3 |
| **row 3** | 0 | 7 | 3 | 1 | 7 | 9 |
| **row 4** | 4 | 8 | 9 | 6 | 0 | 1 |

**Kernel** (3 × 3)

| | col 0 | col 1 | col 2 |
| --- | ---: | ---: | ---: |
| **row 0** | 0 | −2 | 1 |
| **row 1** | 1 | 1 | −1 |
| **row 2** | −1 | −2 | 2 |

### ANSWER Q1 HERE

*(Double-click this cell, replace this line with your answer, then press Shift + Enter.)*

### The image of this project: your photo

Every part of this project processes **a photo you take now with the AR1335 Camera**.
To a computer a grayscale photo is just a grid of numbers, one per pixel, from 0 (black) to
255 (white). Whatever the photo shows, every version does exactly the same amount of work,
so everybody's times are comparable.

The project uses the same **edge-detection kernel** you saw on Mario:

```
-1 -1 -1
-1  8 -1
-1 -1 -1
```

It multiplies the centre pixel by 8 and subtracts its 8 neighbours. Where the brightness
does not change (a wall, the sky) the result is 0; where it changes (the outline of a face,
a letter), the result is large.

Run the cell below. A camera preview opens for 3 seconds, then the Kria takes a
grayscale photo of **1920 x 1080 pixels** and saves it as `photo.pgm`. It does not have to
be your face: point the camera at anything with clear shapes (a hand, a book, a drawing).

To retake the photo, just run the cell again: the new photo replaces the old one.

In [ ]:
image = capture_image()
gray_scale_image = import_photo(image)
notebook_image(image=gray_scale_image)

**Q2.** Your photo is 1920 * 1080 pixels. How many output pixels does the convolution
produce, and how many **multiply-adds** does it need for the whole photo?

### ANSWER Q2 HERE

*(Double-click this cell, replace this line with your answer, then press Shift + Enter.)*

**Q3.** In the output, which parts of your photo turned dark (strong edges) and which
stayed white? Use the kernel to explain why a flat area gives 0. *(Hint: add up the 9
numbers of the kernel.)*

### ANSWER Q3 HERE

*(Double-click this cell, replace this line with your answer, then press Shift + Enter.)*

---
## Part 2. The same loops in Python and C

### Python

Four nested loops: every output pixel, then every kernel position.

```python
for y in range(oh):                  # every output row ...
    for x in range(ow):              # ... every output column
        acc = 0
        for ky in range(3):          # every kernel row
            for kx in range(3):      # every kernel column
                acc += img[y + ky][x + kx] * k[ky][kx]
        out[y][x] = acc
```

Python does not run this directly on the CPU. The **interpreter** reads each operation
while the program runs: it looks up `img`, finds row `y + ky` in a list, finds the element,
checks that both values are numbers, creates a new integer object for the product, and so
on. Each `+` and `*` costs many steps.

Run the cell below to measure the Python version on your photo. Notice that it takes **about 4
seconds** for **18.6 million** multiply-adds, the same work that every version does.

In [ ]:
!python3 conv_bench.py --only Python

### C v1 (`src/conv.c`): column order

For this case, the same four loops will be implemented. However, the difference is *when* the work happens: the compiler (`gcc -O2`)
translates them to machine instructions **once**, before the program runs. Unlike this implementation, Python reads and executes each line at the execution time, not precompiling it. This difference affects performance substantially but affects modularity.

The following code shows how the first C version of the convolution walks the image **column by column**: the outer loop is `x`. For illustrative purposes, the comments show the equivalent RISC-V instructions you would write in
Ripes, with `a0` = `in`, `a1` = `out`, `a2` = `k`, `a4` = `w`, `s0` = `x`, `s1` = `y`,
`s2` = `ky`, `s3` = `kx`, `s4` = `acc`, `s5` = `oh`, `s6` = `ow`. Notice how **one line of
C** becomes **many instructions** in the ASM RISC-V implementation:

```c
void conv_c_cols(int *in, int *out, int *k, int h, int w)
{
    int oh = h - 2;                               // output height
    int ow = w - 2;                               // output width

    for (int x = 0; x < ow; x++) {                //     li   s0, 0           # x = 0
                                                  // loop_x:
        for (int y = 0; y < oh; y++) {            //     li   s1, 0           # y = 0
                                                  // loop_y:
            int acc = 0;                          //     li   s4, 0           # acc = 0
            for (int ky = 0; ky < 3; ky++) {      //     li   s2, 0           # ky = 0
                                                  // loop_ky:
                for (int kx = 0; kx < 3; kx++) {  //     li   s3, 0           # kx = 0
                                                  // loop_kx:
                    acc += in[(y + ky) * w + (x + kx)] * k[ky * 3 + kx];
                                                  //     add  t0, s1, s2      # y + ky
                                                  //     mul  t0, t0, a4      # (y + ky) * w
                                                  //     add  t1, s0, s3      # x + kx
                                                  //     add  t0, t0, t1      # index into in[]
                                                  //     slli t0, t0, 2       # index * 4 bytes
                                                  //     add  t0, a0, t0      # address of the pixel
                                                  //     lw   t2, 0(t0)       # pixel
                                                  //     li   t3, 3
                                                  //     mul  t3, s2, t3      # ky * 3
                                                  //     add  t3, t3, s3      # index into k[]
                                                  //     slli t3, t3, 2       # index * 4 bytes
                                                  //     add  t3, a2, t3      # address of the coefficient
                                                  //     lw   t4, 0(t3)       # coef
                                                  //     mul  t2, t2, t4      # pixel * coef
                                                  //     add  s4, s4, t2      # acc += pixel * coef
                }                                 //     addi s3, s3, 1       # kx++
                                                  //     li   t5, 3
                                                  //     blt  s3, t5, loop_kx # next kx while kx < 3
            }                                     //     addi s2, s2, 1       # ky++
                                                  //     blt  s2, t5, loop_ky # next ky while ky < 3
            out[y * ow + x] = acc;                //     mul  t0, s1, s6      # y * ow
                                                  //     add  t0, t0, s0      # + x
                                                  //     slli t0, t0, 2       # * 4 bytes
                                                  //     add  t0, a1, t0      # address of out[y * ow + x]
                                                  //     sw   s4, 0(t0)       # store acc
        }                                         //     addi s1, s1, 1       # y++
                                                  //     blt  s1, s5, loop_y  # next y while y < oh
    }                                             //     addi s0, s0, 1       # x++
                                                  //     blt  s0, s6, loop_x  # next x while x < ow
}
```

Run the cell below to measure C v1 on the same photo. It runs 5 times and shows the median
of its time. Compare it with the time of Python.

In [ ]:
!python3 conv_bench.py --only "C v1"

**Q4.** How many times faster is C v1 than Python on the Kria? Both run the same four
loops on the same photo. 

### ANSWER Q4 HERE

*(Double-click this cell, replace this line with your answer, then press Shift + Enter.)*

### How Python calls C and assembly

The `make` command turns `src/conv.c` and `src/conv_asm.S` into a set of **shared libraries** (`build/*.so`). The Python script 
`conv_bench.py` loads them with Python's `ctypes` module and calls the functions as if they
were Python functions, passing the image as a block of memory. Python only measures the
time and checks the output; this approach makes the workload runs as native machine code.

---
## Part 3. Measure C v2 in row order

Run the supplied cell as written. It builds the C function and times it on the same photo as C v1. Check that the benchmark reports `OK` before recording its time. The benchmark label `C v2 rows (yours)` refers to the supplied function in this cell.

Follow the two outer loops. C v1 advances down a column between output pixels. C v2 advances across a row. Both compute the same nine products per output pixel.

In [ ]:
from conv_demo import build_c_v2

my_c_v2 = """
void conv_c_rows(int *in, int *out, int *k, int h, int w){
    int oh = h - 2;    // output height
    int ow = w - 2;    // output width

    // Visit adjacent output columns within each row.
    for (int y = 0; y < oh; y++) {
        for (int x = 0; x < ow; x++) {
            int acc = 0;  // Start a new sum for this output pixel.
            for (int ky = 0; ky < 3; ky++) {
                for (int kx = 0; kx < 3; kx++) {
                    // Pair input pixel (y+ky, x+kx) with kernel entry (ky, kx).
                    acc += in[(y + ky) * w + (x + kx)] * k[ky * 3 + kx];
                }
            }
            out[y * ow + x] = acc;  // Store using output width, after all nine products.
        }
    }
}
"""

build_c_v2(my_c_v2)

### Reading the timing

The image is stored row by row. With 32-bit pixels, moving one column right advances 4 bytes. Moving one row down in a 1920-pixel-wide image advances 7680 bytes. A cache line holds several neighbouring pixels, so traversal order affects how soon the processor reuses data it has already fetched.

**Q5.** Record both times using the same unit. Divide the C v1 time by the C v2 time. A ratio above 1 means C v2 finished sooner. How might the distance between successive output writes affect cache use?

### ANSWER Q5 HERE

| Measurement | Your run |
| --- | --- |
| C v1 time (include unit) | |
| C v2 time (same unit) | |
| C v1 time ÷ C v2 time | |

**Observation:** *Write one sentence about your measurement.*

---
## Part 4. From RISC-V (Ripes) to ARM64

Before moving forward, we need to clarify that the Kria KV260 Processing System (CPU) does not have a RISC-V architecture. Instead, it uses an ARM64 architecture and instruction set. Thus, the instruction format and diversity changes drastrically with respect to the RV32I instruction set. Let's pause for a moment to analyze the main differences and similarities of both architectures.

In Ripes you wrote **RISC-V** assembly. The Kria's CPU speaks **ARM64**
(also called AArch64). Both are RISC architectures built on the same ideas: registers,
instructions with a destination and two operands, and memory reached only through loads
and stores. What changes is mostly how things are written. Use these two tables as a
dictionary while you read the ARM64 code. Notice that the **ARM64** architecture has natively implemented the madd (multiply-add) instruction. This is the core instruction used in neural networks and convolutions.

**Registers**

| Use | RISC-V (Ripes) | ARM64 |
| --- | --- | --- |
| Function arguments and result | `a0`–`a7` | `x0`–`x7` |
| Temporaries (free to overwrite) | `t0`–`t6` | `x9`–`x15` (also `x16`, `x17`) |
| Saved (must be restored before returning) | `s0`–`s11` | `x19`–`x28` |
| Return address | `ra` | `x30` (also called `lr`) |
| Stack pointer | `sp` | `sp` |
| Always zero | `zero` | `xzr` / `wzr` |
| Size | 32 bits (RV32) | `xN` = 64 bits; `wN` = the low 32 bits of `xN` |

In this project, pixels are 32-bit numbers, so pixel arithmetic uses `w` registers; memory
addresses and loop counters use `x` registers.

**Instructions**

| What it does | RISC-V (Ripes) | ARM64 | How to read the ARM64 |
| --- | --- | --- | --- |
| Load a constant | <code style="white-space:nowrap">li t0, 3</code> | <code style="white-space:nowrap">mov x15, #3</code> | `x15` = 3. Constants start with `#`. |
| Copy a register | <code style="white-space:nowrap">mv s4, s2</code> | <code style="white-space:nowrap">mov w4, w2</code> | `w4` = `w2`. |
| Add / add a constant | <code style="white-space:nowrap">add t0, t1, t2</code> / <code style="white-space:nowrap">addi t0, t0, 1</code> | <code style="white-space:nowrap">add x0, x1, x2</code> / <code style="white-space:nowrap">add x0, x0, #1</code> | `x0` = `x1` + `x2` / `x0` = `x0` + 1. The same `add` does both. |
| Multiply | <code style="white-space:nowrap">mul t0, t1, t2</code> | <code style="white-space:nowrap">mul w0, w1, w2</code> | `w0` = `w1` × `w2`. |
| Shift left | <code style="white-space:nowrap">slli t0, t0, 2</code> | <code style="white-space:nowrap">lsl x0, x0, #2</code> | `x0` = `x0` << 2, that is `x0` × 4. |
| Load a word from memory | <code style="white-space:nowrap">lw t0, 4(a0)</code> | <code style="white-space:nowrap">ldr w0, [x0, #4]</code> | `w0` = the 4-byte word at address `x0` + 4. Inside `[ ]`: the base address, then the offset in bytes. |
| Store a word | <code style="white-space:nowrap">sw t0, 0(a1)</code> | <code style="white-space:nowrap">str w0, [x1]</code> | Write `w0` (4 bytes) at address `x1`. With no offset, it is `x1` itself. |
| Load, then advance the pointer | <code style="white-space:nowrap">lw t0, 0(a0)</code> + <code style="white-space:nowrap">addi a0, a0, 4</code> | <code style="white-space:nowrap">ldr w0, [x0], #4</code> | `w0` = the word at address `x0`, **then** `x0` = `x0` + 4. The `#4` after `]` is added to the pointer after loading (post-increment), ready for the next word. |
| Load `array[index]` (4-byte words) | <code style="white-space:nowrap">slli</code> + <code style="white-space:nowrap">add</code> + <code style="white-space:nowrap">lw</code> | <code style="white-space:nowrap">ldr w0, [x1, x2, lsl #2]</code> | `w0` = the word at address `x1` + `x2` × 4. `x1` is the address of the array, `x2` the index, and `lsl #2` multiplies the index by 4 because each word is 4 bytes. |
| Branch if less than | <code style="white-space:nowrap">blt t0, t1, loop</code> | <code style="white-space:nowrap">cmp x0, x1</code> then <code style="white-space:nowrap">b.lt loop</code> | `cmp` compares `x0` with `x1` and remembers the result (in the *flags*); `b.lt` jumps to `loop` if it was "less than". |
| Branch if (not) zero | <code style="white-space:nowrap">beqz t0, done</code> / <code style="white-space:nowrap">bnez t0, loop</code> | <code style="white-space:nowrap">cbz x0, done</code> / <code style="white-space:nowrap">cbnz x0, loop</code> | Jump if `x0` is zero / is not zero. |
| Count down and loop | <code style="white-space:nowrap">addi t0, t0, -1</code> + <code style="white-space:nowrap">bnez t0, loop</code> | <code style="white-space:nowrap">subs x0, x0, #1</code> then <code style="white-space:nowrap">b.ne loop</code> | `x0` = `x0` − 1 and set the flags (the `s` in `subs`); `b.ne` jumps if the result was not zero. |
| Jump / return | <code style="white-space:nowrap">j loop</code> / <code style="white-space:nowrap">ret</code> | <code style="white-space:nowrap">b loop</code> / <code style="white-space:nowrap">ret</code> | Always jump to `loop` / return from the function. |
| Multiply and add (MAC) | <code style="white-space:nowrap">mul</code> + <code style="white-space:nowrap">add</code> (RV32IM has no MAC) | <code style="white-space:nowrap">madd w0, w1, w2, w0</code> | `w0` = `w0` + `w1` × `w2`. The 4th register is the one added. |
| Comment | `# ...` | `// ...` | |

Three things to remember: in ARM64 constants start with `#`, memory operands go inside
`[ ]`, and comparisons are separate from branches (`cmp` sets flags, `b.lt` / `b.ne` read
them).

---
## Part 5. Measure ASM v1 with explicit array indexes

ASM v1 follows the four loops in C v2. For each kernel position, it computes the input and coefficient addresses, loads the two values, and adds their product to the current sum.

### Registers

The same roles as the RISC-V comments of C v1 (Part 2), with the names of Part 4:

| Value | RISC-V (Ripes) | ARM64 |
| --- | --- | --- |
| `in`, `out`, `k`, `h`, `w` (arguments) | `a0`–`a4` | `x0`–`x4` |
| `oh`, `ow` | `s5`, `s6` | `x5`, `x6` |
| `x`, `y` | `s0`, `s1` | `x7`, `x8` |
| `ky`, `kx` | `s2`, `s3` | `x10`, `x11` |
| `acc` | `s4` | `w9` |
| pixel, coefficient | `t2`, `t4` | `w14`, `w16` |
| temporaries | `t0`, `t1`, `t3` | `x12`, `x13`, `x15` |

### How the supplied loop fits together

Read it line by line. As in Ripes, the loops start at **labels** with names; each loop ends
with `cmp` (compare the counter with its limit) and `b.lt label` (if less than, jump back to
the label).

| Label | Start of | Jumped back to while |
| --- | --- | --- |
| `loop_y` | an output row (`x = 0`) | `y < oh` |
| `loop_x` | an output pixel (`acc = 0`, `ky = 0`) | `x < ow` |
| `loop_ky` | a kernel row (`kx = 0`) | `ky < 3` |
| `loop_kx` | a kernel column (load the pixel) | `kx < 3` |

The surrounding routine starts the loops and loads the input pixel:

```asm
conv_asm_v1:
        cmp     x3, #3                  // image smaller than the kernel?
        b.lt    done                    //   then there is nothing to do
        cmp     x4, #3
        b.lt    done
        sub     x5, x3, #2              // oh = h - 2
        sub     x6, x4, #2              // ow = w - 2

        mov     x8, #0                  // y = 0
loop_y:
        mov     x7, #0                  //   x = 0
loop_x:
        mov     w9, #0                  //     acc = 0
        mov     x10, #0                 //     ky = 0
loop_ky:
        mov     x11, #0                 //       kx = 0
loop_kx:
        add     x12, x8, x10            //         y + ky
        mul     x12, x12, x4            //         (y + ky) * w
        add     x13, x7, x11            //         x + kx
        add     x12, x12, x13           //         index into in[]
        ldr     w14, [x0, x12, lsl #2]  //         pixel = in[index]   (lsl #2: * 4 bytes)
```

After the output store, the surrounding routine advances to the next column or row:

```asm
        add     x7, x7, #1              //     x++
        cmp     x7, x6
        b.lt    loop_x                  //     next x while x < ow
        add     x8, x8, #1              //   y++
        cmp     x8, x5
        b.lt    loop_y                  //   next y while y < oh
done:
        ret
```

Run the supplied cell. The helper inserts the code below between those two excerpts, builds the full routine, and measures it. Record the time after the output check reports `OK`.

In [ ]:
from conv_demo import build_v1

my_v1 = """
        mov     x15, #3  // Kernel row width in coefficients
        mul     x15, x10, x15  // Offset of kernel row ky
        add     x15, x15, x11  // Add column kx
        ldr     w16, [x2, x15, lsl #2]  // Load k[ky*3+kx]; each coefficient is four bytes

        mul     w14, w14, w16  // Multiply the input pixel by its coefficient
        add     w9, w9, w14  // Accumulate the product for this output pixel

        add     x11, x11, #1            //         kx++
        cmp     x11, #3
        b.lt    loop_kx                 //         next kx while kx < 3
        add     x10, x10, #1            //       ky++
        cmp     x10, #3
        b.lt    loop_ky                 //       next ky while ky < 3

        mul     x12, x8, x6  // Output row offset: y * output width
        add     x12, x12, x7  // Add output column x
        str     w9, [x1, x12, lsl #2]  // Store the completed nine-product sum
"""

build_v1(my_v1)

**Q6.** Record both times using the same unit. Divide the C v2 time by the ASM v1 time. A ratio above 1 means ASM v1 finished sooner. What happened to elapsed time when the same four loops used explicit assembly indexing?

### ANSWER Q6 HERE

| Measurement | Your run |
| --- | --- |
| C v2 time (include unit) | |
| ASM v1 time (same unit) | |
| C v2 time ÷ ASM v1 time | |

**Observation:** *Write one sentence about your measurement.*

---
## Part 6. Measure ASM v2 with row pointers

v1 recomputes `(y + ky) * w + (x + kx)` with multiplications for every single multiply-add,
and reloads the kernel from memory every time. v2 instead:

- loads the 9 kernel numbers **once** into registers `w9`–`w17`;
- keeps **three pointers** (`x7`, `x8`, `x2`) on the three image rows under the kernel, and
  moves them 4 bytes to the right after each output pixel;
- writes out the 9 kernel positions one after another (no `ky`, `kx` loops).

`ldr w20, [x7], #4` loads a pixel **and** moves the pointer to the next one, in a single
instruction (*post-increment*). One output pixel of v2:

```asm
2:      ldr     w20, [x7], #4           // row y:   in[y][x], then p0 += 4
        mul     w19, w20, w9            //   acc  = pixel * k[0]
        ldr     w20, [x7]               //          in[y][x+1]
        mul     w20, w20, w10           //   product = pixel * k[1]
        add     w19, w19, w20           //   acc += product
        ...                             //   (the same for the other 7 positions)
        str     w19, [x1], #4           // *out++ = acc
        subs    x6, x6, #1              // columns left--
        b.ne    2b                      // Repeat for the next output column
```

Measure ASM v2 on your photo and compare it with the ASM v1 time from Part 5:

In [ ]:
!python3 conv_bench.py --only "ASM v2"

**Q7.** Record both times using the same unit. Divide the ASM v1 time by the ASM v2 time. A ratio above 1 means ASM v2 finished sooner. Which repeated work does v2 remove from the inner loop?

### ANSWER Q7 HERE

| Measurement | Your run |
| --- | --- |
| ASM v1 time (include unit) | |
| ASM v2 time (same unit) | |
| ASM v1 time ÷ ASM v2 time | |

**Observation:** *Write one sentence about your measurement.*

---
## Part 7. Measure ASM v3 with multiply-accumulate

ASM v3 uses the same three row pointers as v2. Follow the accumulator `w19` in the supplied cell. The first product starts the sum with `mul`. Each of the other eight coefficients contributes one product through `madd`:

```asm
madd w19, w20, w10, w19    // new sum = (current pixel × coefficient) + old sum
```

The helper supplies the surrounding loops and output store. Run the cell, confirm the output check reports `OK`, and compare the measured time with ASM v2. One instruction can take multiple cycles; the timing tells you how the change affects the whole convolution.

In [ ]:
from conv_demo import build_v3

my_pixel = """
        // Kernel row 0: advance its pointer once, then read the two neighbours.
        ldr     w20, [x7], #4     // Input row y+0, column x+0
        mul     w19, w20, w9            // Start the sum with coefficient 0
        ldr     w20, [x7]         // Input row y+0, column x+1
        madd    w19, w20, w10, w19      // Add pixel x coefficient 1
        ldr     w20, [x7, #4]     // Input row y+0, column x+2
        madd    w19, w20, w11, w19      // Add pixel x coefficient 2

        // Kernel row 1: advance its pointer once, then read the two neighbours.
        ldr     w20, [x8], #4     // Input row y+1, column x+0
        madd    w19, w20, w12, w19      // Add pixel x coefficient 3
        ldr     w20, [x8]         // Input row y+1, column x+1
        madd    w19, w20, w13, w19      // Add pixel x coefficient 4
        ldr     w20, [x8, #4]     // Input row y+1, column x+2
        madd    w19, w20, w14, w19      // Add pixel x coefficient 5

        // Kernel row 2: advance its pointer once, then read the two neighbours.
        ldr     w20, [x2], #4     // Input row y+2, column x+0
        madd    w19, w20, w15, w19      // Add pixel x coefficient 6
        ldr     w20, [x2]         // Input row y+2, column x+1
        madd    w19, w20, w16, w19      // Add pixel x coefficient 7
        ldr     w20, [x2, #4]     // Input row y+2, column x+2
        madd    w19, w20, w17, w19      // Add pixel x coefficient 8
"""

build_v3(my_pixel)

**Q8.** Record both times using the same unit. Divide the ASM v2 time by the ASM v3 time. A ratio above 1 means ASM v3 finished sooner. How did combining multiply and add affect the measured time?

### ANSWER Q8 HERE

| Measurement | Your run |
| --- | --- |
| ASM v2 time (include unit) | |
| ASM v3 time (same unit) | |
| ASM v2 time ÷ ASM v3 time | |

**Observation:** *Write one sentence about your measurement.*

---
## Part 8. ASM v4: NEON, 4 pixels at once

The Kria KV260's CPU has **NEON** vector registers (`v0`–`v31`) of 128 bits, which hold
4 numbers of 32 bits each. One vector instruction works on all 4 at the same time:

```asm
ldr     q16, [x7]                   // load 4 pixels: in[y][x .. x+3]
mla     v20.4s, v16.4s, v0.s[1]     // 4 accumulators += 4 pixels * k[1]
```

Each lane accumulates one output: columns `x`, `x+1`, `x+2`, and `x+3` of the same row. The four lanes share a coefficient at each kernel position. Run the supplied benchmark and compare its time with the scalar ASM v3 run.

In [ ]:
!python3 conv_bench.py --only "ASM v4"

**Q9.** Record both times using the same unit. Divide the ASM v3 time by the ASM v4 time. A ratio above 1 means ASM v4 finished sooner. How does your measured ratio compare with the four output pixels processed per vector iteration?

### ANSWER Q9 HERE

| Measurement | Your run |
| --- | --- |
| ASM v3 time (include unit) | |
| ASM v4 time (same unit) | |
| ASM v3 time ÷ ASM v4 time | |

**Observation:** *Write one sentence about your measurement.*

---
## Part 9. Measure all the versions

You have measured each version on its own. Now compare them all in one table, under the
same conditions: the cell below runs every version on your photo, with the
edge-detection kernel. Python runs once (**about 9 seconds**); each C and assembly version
runs 5 times and the median is shown. Every output is compared with Python's: `OK` means identical.
This table sums up the whole project: use its numbers in your conclusions (Part 11).

Close other programs before running it.

In [ ]:
!python3 conv_bench.py

---
## Part 10. The race

Now see it live, on your photo. Python processes it on a **CPU core of its own**. Meanwhile,
on another core, the six C and assembly versions process the same photo **one after
another**, never at the same time, so no version slows another one down.

- **Panel 1 (one photo):** the progress of the photo for every version. The C and assembly
  bars fill one after another in less than a second, each with its time, while Python's
  bar crawls for about 9 seconds.
- **Panel 2:** how many photos each C and assembly version **could** process in the time
  Python has used so far. It is calculated, not run: *Python's time ÷ that version's time
  for one photo*. It grows while Python works. When the longest bar reaches the end, the
  scale grows (*full bar = N photos*) and all bars shrink together.

Run the cell and watch the bars (about 15 seconds including the preparation).

In [ ]:
race = notebook_race()

---
## Part 11. Run your Arm C function on MicroBlaze

In the full-adder exercise, Python on the Arm set three inputs, and the circuit in the FPGA produced the sum and carry. In the Arm race, the processor itself computed each output pixel. Python, compiled C, and AArch64 assembly all ran on the same Arm hardware. Changing the loop order or using NEON changed how that processor did the work.

Here you keep your **C v2 row-order function** and change the processor that executes it. The KV260 has Arm cores built into the chip. MicroBlaze V is a **soft processor** built in the FPGA’s programmable logic. Vivado defines that processor and its connections to memory. Vitis compiles your C loop into RISC-V instructions for the MicroBlaze V to execute. Both processors run the convolution as software. A dedicated convolution circuit would instead implement the filter’s multiplications and additions directly in logic.

PYNQ on the Arm starts the MicroBlaze and reads its results through shared memory. Check that both runs produce the same pixels before comparing times. A PYNQ timer started just before sending `START` and stopped when it observes `DONE` includes the command and polling overhead. Count image transfers only if they occur between those timestamps.

Finish the Arm race and record the **C v2 rows (yours)** time from Part 9. If you started the live video, run the stop cell above. Keep the same 1920 × 1080 benchmark photo for the MicroBlaze comparison.

**1. Create the Vitis workspace on your development computer.** Open a terminal on the computer or VM with Vitis installed. From the repository directory that contains the workshop script, run

```bash
source create_vitis_workspace.sh
```

When the script finishes successfully, open the generated workspace in Vitis and select the MicroBlaze application.

**2. Copy your tested function from Jupyter.** Check that the supplied Part 3 cell reported `OK`. Run the cell below to print that same C source.

In [ ]:
print(my_c_v2.strip())

**3. Paste the function into the MicroBlaze application.** Copy the complete output, from `void conv_c_rows(...)` through its closing brace. In the application source, place the definition above `main()`. Ensure the pictures bytes have been given in an `.inc` to use in the convolution.

**4. Call it from the convolution step.** Inside the GPIO start handler, call the function after the input is ready and before signaling completion. The argument order is `conv_c_rows(input, output, kernel, height, width)`. Use the application's actual buffer names. The function expects 32-bit integer pixels and nine contiguous integer coefficients. For a packed tile, pass that tile’s height and width.

Build the application in Vitis and load the resulting ELF onto the MicroBlaze in Vivado for the matching hardware design. These steps are described in the presentation for further clarity. Copy the `.bit and .hwh` files as in practice 1 on the **Overlays** folder. Run the filter on the same photo and compare its signed output with the Arm result before displaying the edges.

In [ ]:
from microblaze import load_microblaze, run_tiled_convolution

cleanup_device_trees()

if overlay is not None:
    overlay.free()
    overlay = None

bitstream_name = "your_design.bit"  # Replace with your filename
overlay = None
overlay = load_microblaze(Path("Overlays") / bitstream_name)
print(f"Loaded {bitstream_name}")

In [ ]:
from PIL import Image as PILImage
from time import perf_counter
import numpy as np

with PILImage.open(gray_scale_image) as pgm:
    photo = np.array(pgm.convert("L"), dtype=np.uint8)

start = perf_counter
edges = run_tiled_convolution(overlay, photo)
elapsed = perf_counter() - start

print(f"Execution took {elapsed:.6f} seconds")

---
## Part 12. Compare YOLO object detection on the Arm CPU and the DPU

In the Arm race, you changed the code that applied a 3 × 3 edge filter to your photo. In this exercise, you keep the detection task fixed and change where inference runs. You will process the same video with **YOLO** on the KV260’s **Arm CPU** and its **Deep Learning Processing Unit (DPU)**. For each frame, YOLO predicts which objects are present and where they are. The script displays those predictions as labeled bounding boxes.

In the **CPU run**, the Arm processor executes the model. In the **DPU run**, the provided Python script uses the supplied DPU overlay and `.xclbin` to run supported model operations on the FPGA accelerator. The Arm still coordinates the run and handles tasks such as preparing frames and drawing boxes. All hardware files are provided for this exercise.

### Run the comparison

1. Run the provided Python script using the video from the camera and the YOLO model. Follow the script’s controls to obtain results for both the CPU and the DPU.
2. Compare performance using the **same metric** for both runs. For inference time per frame, divide **CPU time by DPU time**. For throughput, divide **DPU frames per second by CPU frames per second**.

**Discuss:** If DPU inference takes half as long per frame, would you expect the displayed video to run twice as fast? Consider which timed operations move to the DPU and which still run on the Arm.


In [ ]:
from yolo_capture_pipeline import notebook_stream

stream = notebook_stream()